# 🗂 Dataset 設定ファイルの深掘り

量子化設定・template の実体・auto_map・recipe.yaml。


In [ ]:
# Dataset の設定ファイル深掘り（30 秒）
import json
import os

ROOT = ("/kaggle/input/datasets/yamahei21python/"
        "qwen38-27b-uncensored-hauhaucs-awq-mtp")
cfg = json.load(open(os.path.join(ROOT, "config.json")))
print("keys:", sorted(cfg.keys()))
print("\nquantization_config:")
print(json.dumps(cfg.get("quantization_config"), indent=2, ensure_ascii=False))
print("\nauto_map:", cfg.get("auto_map"))
print("rope:", cfg.get("rope_scaling"), "| rope_theta:", cfg.get("rope_theta"))
print("sliding_window:", cfg.get("sliding_window"),
      "| layer_types:", (cfg.get("layer_types") or [])[:6])
print("num_experts:", cfg.get("num_experts"),
      "| moe_intermediate:", cfg.get("moe_intermediate_size"))
print("intermediate_size:", cfg.get("intermediate_size"))

# tokenizer_config の chat_template と jinja ファイルの一致
tk = json.load(open(os.path.join(ROOT, "tokenizer_config.json")))
embedded = tk.get("chat_template")
print("\ntokenizer_config chat_template:", type(embedded).__name__,
      len(embedded) if isinstance(embedded, str) else embedded)
tpl_files = ["chat_template.jinja", "chat_template_medium.jinja"]
for name in tpl_files:
    p = os.path.join(ROOT, name)
    if not os.path.isfile(p):
        continue
    body = open(p, encoding="utf-8").read()
    print(f"{name}: {len(body)} chars | == tokenizer_config: "
          f"{body.strip() == (embedded or '').strip()}")
    for kw in ("reasoning_effort", "thinking", "tool_call", "xhigh"):
        print(f"   contains {kw}:", kw in body)

print("\ntokenizer_config keys:", sorted(tk.keys()))
print("bos/eos:", tk.get("bos_token"), tk.get("eos_token"),
      "| model_max_length:", tk.get("model_max_length"))

# recipe.yaml（vendor の変換情報）
p = os.path.join(ROOT, "recipe.yaml")
if os.path.isfile(p):
    lines = open(p, encoding="utf-8", errors="replace").read().splitlines()
    print(f"\nrecipe.yaml: {len(lines)} lines")
    for line in lines[:40]:
        print("  ", line[:150])

# 余計なファイルが vLLM を混乱させないか
print("\nall files:")
for f in sorted(os.listdir(ROOT)):
    fp = os.path.join(ROOT, f)
    print(f"  {f:35s} {'DIR' if os.path.isdir(fp) else os.path.getsize(fp)}")
